# Epsilon Fund — Strategy Testing
---

In [ ]:
import pandas as pd
import numpy as np
import sys

# ── Set your repo root path ────────────────────────────────────────────────────
ROOT = r'/Users/justiniturregui/Desktop/epsilon/github/Epsilon-Quant-Research'  # ← change to yours
# ──────────────────────────────────────────────────────────────────────────────

sys.path.append(ROOT + '/infrastructure/data')
sys.path.append(ROOT + '/infrastructure/backtester')

from binance_client import get_binance_client, get_data, get_multiple_data
from engine import backtest

---
## Data

**Pairs** — any Binance pair in `BASEQUOTE` format (e.g. `BTCUSDT`, `ETHUSDT`, `SOLUSDT`, `BNBUSDT`).  
Verify availability at [binance.com/en/trade](https://www.binance.com/en/trade).

**Intervals** — `'1m'` `'5m'` `'15m'` `'1h'` `'4h'` `'1d'` `'1w'`

**Lookback** — days of history: `365` (1y) · `730` (2y) · `1825` (5y) · `2555` (7y, recommended minimum)

In [ ]:
SYMBOL   = 'BTCUSDT'
INTERVAL = '1d'
LOOKBACK = 2555

# ── Multiple pairs (optional) ──────────────────────────────────────────────────
# SYMBOLS = ['BTCUSDT', 'ETHUSDT', 'SOLUSDT']
# data_dict = get_multiple_data(client, SYMBOLS, INTERVAL, LOOKBACK)
# Access via: data_dict['BTCUSDT_1d'], data_dict['ETHUSDT_1d'] ...
# ──────────────────────────────────────────────────────────────────────────────

client = get_binance_client()
df = get_data(client, SYMBOL, INTERVAL, LOOKBACK)

print(f'{SYMBOL} | {INTERVAL} | {df.index[0].date()} -> {df.index[-1].date()} | {len(df)} rows')
df.tail(3)

BTCUSDT | 1d | 2019-03-06 -> 2026-03-03 | 2555 rows


,Open,High,Low,Close,Volume
Time,,,,,
2026-03-01,66973.26,68199.99,65056.00,65776.47,23201.85859
2026-03-02,65776.48,70096.00,65259.21,68830.06,32010.38431
2026-03-03,68830.06,69258.08,66158.00,67674.01,16980.98382


---
## Strategy

**Available columns:** `Open` `High` `Low` `Close` `Volume`

**Required output:** a `position` column — `1` long · `0` flat · `-1` short  
**Optional:** `position_size` column (0–1) to use fractional capital

> Signals are shifted 1 bar by the engine — no need to shift manually.

In [ ]:
strategy_df = df.copy()

# ── Strategy logic ─────────────────────────────────────────────────────────────

# --- REGIME FILTER (computed on daily bars resampled from hourly) ---
daily = pd.DataFrame()
daily['open']  = strategy_df['Open'].resample('1D').first()
daily['high']  = strategy_df['High'].resample('1D').max()
daily['low']   = strategy_df['Low'].resample('1D').min()
daily['close'] = strategy_df['Close'].resample('1D').last()
daily = daily.dropna()

MA_L  = 20
ATR_N = 14
K     = 1.5

daily['MA']    = daily['close'].rolling(MA_L).mean()

# ATR on daily bars
high       = daily['high']
low        = daily['low']
close      = daily['close']
prev_close = close.shift(1)
tr         = pd.concat([(high - low), (high - prev_close).abs(), (low - prev_close).abs()], axis=1).max(axis=1)
daily['ATR']   = tr.rolling(ATR_N).mean()

daily['upper'] = daily['MA'] + 1.5 * K * daily['ATR']
daily['lower'] = daily['MA'] - 0.5 * K * daily['ATR']

# State machine: 1 = bull, 0 = bear
regime = pd.Series(index=daily.index, dtype='float64')
start_idx = daily.dropna(subset=['upper', 'lower']).index.min()

state = 1 if daily.loc[start_idx, 'close'] > daily.loc[start_idx, 'MA'] else 0

for t in daily.loc[start_idx:].index:
    c  = float(daily.loc[t, 'close'])
    up = float(daily.loc[t, 'upper'])
    lo = float(daily.loc[t, 'lower'])
    if state == 0 and c > up:
        state = 1
    elif state == 1 and c < lo:
        state = 0
    regime.loc[t] = state

daily['regime_raw'] = regime
daily['regime']     = daily['regime_raw'].shift(1)  # ← 1-bar shift: no lookahead

# Map daily regime back to hourly index
regime_hourly = daily['regime'].reindex(strategy_df.index, method='ffill')

# --- MA CROSSOVER (computed on hourly bars) ---
fast = 20
slow = 190

strategy_df['MA_fast'] = strategy_df['Close'].rolling(fast).mean()
strategy_df['MA_slow'] = strategy_df['Close'].rolling(slow).mean()

bull  = (regime_hourly == 1)
enter = (strategy_df['MA_fast'] > strategy_df['MA_slow']) & bull

# --- POSITION LOOP ---
trail = 0.5   # exit if price drops 50% from peak since entry

strategy_df['position'] = 0

in_pos = 0
peak   = np.nan

for t in strategy_df.index:
    # Skip bars where indicators haven't warmed up yet
    if pd.isna(strategy_df.loc[t, 'MA_fast']) or pd.isna(strategy_df.loc[t, 'MA_slow']):
        continue
    if pd.isna(regime_hourly.loc[t]):
        continue

    if in_pos == 0:
        if bool(enter.loc[t]):
            in_pos = 1
            peak   = float(strategy_df.loc[t, 'Close'])

    else:
        c = float(strategy_df.loc[t, 'Close'])
        if c > peak:
            peak = c

        trail_hit = c < (1.0 - trail) * peak

        if (not bool(bull.loc[t])) or trail_hit:
            in_pos = 0
            peak   = np.nan

    strategy_df.loc[t, 'position'] = in_pos

# ──────────────────────────────────────────────────────────────────────────────

# Drop only the indicator warmup rows — preserve all flat (0) position bars
strategy_df = strategy_df.dropna(subset=['MA_fast', 'MA_slow'])
strategy_df['position'] = strategy_df['position'].fillna(0).astype(int)

strategy_df['position'].value_counts()

position
 1    1372
 0    1180
-1       3
Name: count, dtype: int64

---
## Backtest

| Parameter | Options | Default |
|---|---|---|
| `cost` | Cost per trade as decimal — `0.001` = 0.1% | `0.0` |
| `show_plot` | `True` / `False` | `True` |
| `save_html` | Filename string or `None` | `None` |
| `show_trades` | Overlay entry/exit markers on price chart | `False` |
| `benchmark_data` | DataFrame with `Close` column for buy & hold comparison | same asset |

In [ ]:
results = backtest(
    data           = strategy_df,
    cost           = 0.0,
    show_plot      = True,
    save_html      = None,
    show_trades    = False,
    benchmark_data = None
)

print(f"Return        {results['total_return']*100:>8.2f}%")
print(f"Sharpe        {results['sharpe_ratio']:>8.2f}")
print(f"Max Drawdown  {results['max_drawdown']*100:>8.2f}%")
print(f"Profit Factor {results['profit_factor']:>8.2f}")
print(f"Win Rate      {results['win_rate']*100:>8.2f}%")
print(f"# Trades      {results['num_trades']:>8}")

Return         2177.48%
Sharpe            1.21
Max Drawdown    -60.44%
Profit Factor     3.52
Win Rate         38.00%
# Trades            50
